In [ ]:
import os
import json
import csv
from dataclasses import dataclass, asdict
from typing import Dict, Tuple, Literal, Any

import numpy as np
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm import tqdm

import datasets
import unet
from SplitNet import SplitNet
import unetFixed

# REQUIRED FILES

# datasets.py
# unetFixed.py
# SplitNet.py
# unet.py
# Also requires the simulation PNG files.
# The relative location of that folder is set inside datasets.py.


# USAGE

# run_experiment_grid() runs every combination of:
#
#   model_types × dataset_modes × training_modes × darcy_weights
#
# For training_mode="baseline_full", darcy_weight is automatically set to 0.0
#
# Common model_types:
#   "splitnet_attn", "splitnet", "unet", "attn_unet", "unetFixed"
#
# Common dataset_modes:
#   "fixed", "border"
#
# training_modes:
#   "physics_limited"   : limited/masked supervision + Darcy loss
#   "baseline_full"     : full-image supervision, no Darcy


# EXAMPLE: official Darcy models
"""
results = run_experiment_grid(
    model_types=("splitnet_attn", "splitnet", "unet", "attn_unet", "unetFixed"),
    dataset_modes=("fixed", "border"),
    training_modes=("physics_limited",),
    darcy_weights=(0.1, 1.0, 5.0, 10.0),
    epochs=20,
    sim_max_exclusive=100,
    base_save_dir="official_darcy/final",
)


# EXAMPLE: baseline full models
baseline_results = run_experiment_grid(
    model_types=("splitnet_attn", "attn_unet", "unetFixed"),
    dataset_modes=("fixed", "border"),
    training_modes=("baseline_full",),
    epochs=20,
    sim_max_exclusive=100,
    base_save_dir="baseline_full/final",
)


# EXAMPLE: custom sensor layout and step range
custom_results = run_experiment_grid(
    model_types=("attn_unet",),
    dataset_modes=("fixed",),
    training_modes=("physics_limited",),
    darcy_weights=(0.1, 1.0),
    epochs=50,
    sim_max_exclusive=100,
    base_save_dir="custom_runs",
    dataset_kwargs={
        "points_per_side": 5,
        "radius": 3,
        "steps": (0, 100),
    },
)
"""

In [ ]:
# Reproducibility and device
# ---------------------------------------------

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Configuration
# ---------------------------------------------

ModelType = Literal["splitnet_attn", "splitnet", "unet", "attn_unet", "unetfixed"]
DatasetMode = Literal["border", "fixed"]
TrainingMode = Literal["physics_limited", "baseline_full"]

@dataclass
class ExperimentConfig:
    model_type: ModelType = "splitnet_attn"
    dataset_mode: DatasetMode = "border"

    # physics_limited:
    #   Uses Limited datasets: (input_sample, sparse_target, mask)
    # baseline_full:
    #   Uses Full datasets: (input_sample, full_target)
    training_mode: TrainingMode = "physics_limited"

    mse_weight: float = 1.0
    darcy_weight: float = 1.0

    epochs: int = 50
    batch_size: int = 8
    lr: float = 1e-3

    channels: str = "KP"
    train_sims_path: str = "../train_sims.npy"
    val_sims_path: str = "../val_sims.npy"
    sim_max_exclusive: int = 500

    save_prefix: str = None
    save_best: bool = True

    # Folder where experiment data is saved
    # If save_prefix is given, files are saved using that prefix otherwise, save_dir/run_name is used instead
    save_dir: str = "minimum_info/results"
    run_name: str = None


    # Used to decide "best" model checkpoint
    # If None, run_experiment chooses a default:
    #     physics_limited -> val_supervised_mse
    #     baseline_full    -> val_total_mse
    best_metric: str = None

    # Optional dataset overrides, e.g.
    # dataset_kwargs={"points_per_side": 5, "radius": 3, "steps": (0, 200)}
    dataset_kwargs: Dict[str, Any] = None


# Darcy utilities
# ---------------------------------------------

def darcy_loss_from_output(out: torch.Tensor) -> torch.Tensor:
    """
    Computes div(K * grad(P))

    Expected shape: [batch size, chanels, height, width]
    Requires at least 2 output channels: channel 0 = K, channel 1 = P.

    Returns a scalar Darcy loss
    """
    if out.shape[1] < 2:
        raise ValueError("Darcy residual requires at least 2 output channels: K and P.")

    k = out[:, 0:1]
    p = out[:, 1:2]

    p_y, p_x = torch.gradient(p, dim=(-2, -1))

    flux_y = k * p_y
    flux_x = k * p_x

    div_y = torch.gradient(flux_y, spacing=(1,), dim=(-2,))[0]
    div_x = torch.gradient(flux_x, spacing=(1,), dim=(-1,))[0]

    return ((div_y + div_x)**2).mean()


# Model factory
# ---------------------------------------------

def make_model(model_type: ModelType = "splitnet_attn", channels: str = "KP") -> nn.Module:
    """
    Builds a model with the selected channel setup

    channels='KP': models output 2 channels: K and P.
    SplitNet outputs K and P, it is only compatible with channels='KP'.
    """
    model_type = model_type.lower()

    if channels == "all":
        num_channels = 3
    elif channels == "KP":
        num_channels = 2
    elif channels in ["K", "P", "phi"]:
        num_channels = 1
    else:
        raise ValueError("channels must be 'all', 'KP', 'K', 'P', or 'phi'.")

    if model_type == "splitnet_attn":
        if channels != "KP":
            raise ValueError("SplitNet is designed for channels='KP'.")
        return SplitNet(attn=True).to(DEVICE)

    if model_type == "splitnet":
        if channels != "KP":
            raise ValueError("SplitNet is designed for channels='KP'.")
        return SplitNet(attn=False).to(DEVICE)

    if model_type == "unet":
        return unet.SmallUnet(channels=num_channels).to(DEVICE)

    if model_type == "attn_unet":
        return unet.AttnUnet(channels=num_channels).to(DEVICE)
    
    if model_type == "unetfixed":
        return unetFixed.UNet(channels=num_channels).to(DEVICE)

    raise ValueError(f"Unknown model_type: {model_type}")


# Dataset loader
# ---------------------------------------------

def _dataset_class(dataset_mode: DatasetMode, training_mode: TrainingMode):
    """
    Chooses the dataset class for the experiment

    training_mode='physics_limited':
        Uses Limited datasets.
    training_mode='baseline_full':
        Uses Full datasets.
    """
    if training_mode == "physics_limited":
        if dataset_mode == "border":
            return datasets.BorderDenseDatasetLimited
        if dataset_mode == "fixed":
            return datasets.FixedDenseDatasetLimited

    if training_mode == "baseline_full":
        if dataset_mode == "border":
            return datasets.BorderDenseDatasetFull
        if dataset_mode == "fixed":
            return datasets.FixedDenseDatasetFull

    raise ValueError(
        "Invalid dataset options. dataset_mode must be 'border', 'fixed'; "
        "training_mode must be 'physics_limited' or 'baseline_full'."
    )


def load_sim_ids(path: str, sim_max_exclusive: int = 500) -> np.ndarray:
    sims = np.load(path)
    if sim_max_exclusive is not None:
        sims = sims[sims < sim_max_exclusive]
    return sims

class ChannelSelectDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, channels="KP"):
        self.base_dataset = base_dataset
        self.channels = channels

    def _idx(self):
        if self.channels == "all":
            return [0, 1, 2]
        if self.channels == "KP":
            return [0, 1]
        if self.channels == "K":
            return [0]
        if self.channels == "P":
            return [1]
        if self.channels == "phi":
            return [2]
        raise ValueError("Bad channels")

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        chans = self._idx()

        if len(item) == 3:
            feat, label, mask = item
            return feat[chans], label[chans], mask

        feat, label = item
        return feat[chans], label[chans]


def make_loaders(config: ExperimentConfig) -> Tuple[DataLoader, DataLoader]:
    train_sims = load_sim_ids(config.train_sims_path, config.sim_max_exclusive)
    val_sims = load_sim_ids(config.val_sims_path, config.sim_max_exclusive)

    dataset_cls = _dataset_class(config.dataset_mode, config.training_mode)

    kwargs = dict(config.dataset_kwargs or {})
    kwargs["channels"] = config.channels

    train_data = dataset_cls(train_sims, **kwargs)
    val_data = dataset_cls(val_sims, **kwargs)

    train_data = ChannelSelectDataset(train_data, channels=config.channels)
    val_data = ChannelSelectDataset(val_data, channels=config.channels)

    train_loader = DataLoader(train_data, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=config.batch_size, shuffle=False)

    return train_loader, val_loader


# Saving helpers
# ---------------------------------------------

def make_run_prefix(config: ExperimentConfig) -> str:
    """
    Returns the path prefix for all saved files from a run.
    Example prefix: minimum_info/results/fixed_physics_limited_splitnet_attn_darcy_1p0
    """
    if config.save_prefix:
        return config.save_prefix

    if config.run_name:
        name = config.run_name
    else:
        safe_w = str(config.darcy_weight).replace(".", "p")
        name = f"{config.dataset_mode}_{config.training_mode}_{config.model_type}_darcy_{safe_w}"

    return os.path.join(config.save_dir, name)


def ensure_parent_dir(path_prefix: str):
    folder = os.path.dirname(path_prefix)
    if folder:
        os.makedirs(folder, exist_ok=True)


def save_json(path: str, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def save_history_csv(path: str, history: Dict[str, Any]):
    """
    Saves epoch by epoch curves to CSV

    Use later for plotting train/val curves.
    """
    curve_keys = [
        "train_loss_used",
        "train_total_mse",
        "train_supervised_mse",
        "train_mask_mse",
        "train_nonmask_mse",
        "train_darcy",
        "val_total_mse",
        "val_supervised_mse",
        "val_mask_mse",
        "val_nonmask_mse",
        "val_darcy",
    ]

    n_epochs = len(history["train_loss_used"])

    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch"] + curve_keys)

        for i in range(n_epochs):
            row = [i + 1]
            for key in curve_keys:
                values = history.get(key, [])
                row.append(values[i] if i < len(values) else "")
            writer.writerow(row)


def make_run_summary(history: Dict[str, Any], config: ExperimentConfig, best_epoch: int, best_val_score: float) -> Dict[str, Any]:
    """
    Saves one compact summary per run.
    Useful for comparing runs without loading every curve.
    """
    def best(key):
        values = history[key]
        return min(values) if len(values) else None

    def final(key):
        values = history[key]
        return values[-1] if len(values) else None

    summary = {
        "run_name": config.run_name,
        "model_type": config.model_type,
        "dataset_mode": config.dataset_mode,
        "training_mode": config.training_mode,
        "channels": config.channels,
        "mse_weight": config.mse_weight,
        "darcy_weight": config.darcy_weight,
        "epochs": config.epochs,
        "batch_size": config.batch_size,
        "lr": config.lr,
        "best_epoch": best_epoch,
        "best_val_score_used": best_val_score,
        "best_train_loss_used": best("train_loss_used"),
        "best_val_total_mse": best("val_total_mse"),
        "best_val_supervised_mse": best("val_supervised_mse"),
        "best_val_mask_mse": best("val_mask_mse"),
        "best_val_nonmask_mse": best("val_nonmask_mse"),
        "best_val_darcy": best("val_darcy"),
        "final_train_loss_used": final("train_loss_used"),
        "final_val_total_mse": final("val_total_mse"),
        "final_val_supervised_mse": final("val_supervised_mse"),
        "final_val_mask_mse": final("val_mask_mse"),
        "final_val_nonmask_mse": final("val_nonmask_mse"),
        "final_val_darcy": final("val_darcy"),
        "config": asdict(config),
    }

    return summary


def save_run_outputs(path_prefix: str, model: nn.Module, history: Dict[str, Any], config: ExperimentConfig, best_epoch: int, best_val_score: float):
    """
    Saves all analysis outputs for a single experiment

    Files created:
        *_history.csv       epoch curves for comparison
        *_history.pt        full history dict
        *_summary.json      compact final/best metrics + config
        *_final_state.pt    model weights
    """
    ensure_parent_dir(path_prefix)

    save_history_csv(f"{path_prefix}_history.csv", history)
    torch.save(history, f"{path_prefix}_history.pt")

    summary = make_run_summary(history, config, best_epoch, best_val_score)
    save_json(f"{path_prefix}_summary.json", summary)

    torch.save(model.state_dict(), f"{path_prefix}_final_state.pt")


# Loss and metric helpers
# ---------------------------------------------



def align_channels(label: torch.Tensor, out: torch.Tensor) -> torch.Tensor:
    """
    Keeps label channels compatible with model output channels.
    If a dataset gives 3 channels but the model outputs KP only, keep K and P.
    """
    if label.shape[1] == out.shape[1]:
        return label
    if label.shape[1] > out.shape[1]:
        return label[:, : out.shape[1]]
    raise ValueError(f"Label has {label.shape[1]} channels but output has {out.shape[1]} channels.")


def expand_mask_for_channels(mask: torch.Tensor, out: torch.Tensor) -> torch.Tensor:
    if mask.dim() == 3:
        mask = mask.unsqueeze(1)
    return mask.expand(-1, out.shape[1], -1, -1)


def mse_on_region(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """
    Divides by number of masked pixels, not whole image size
    """
    mask = mask.float()
    diff2 = ((pred - target) ** 2) * mask
    denom = mask.sum()
    if denom.item() == 0:
        return torch.tensor(0.0, device=pred.device)
    return diff2.sum() / denom


def supervised_loss(
    out: torch.Tensor,
    label: torch.Tensor,
    mask: torch.Tensor,
    training_mode: TrainingMode,
    crit: nn.Module,
) -> torch.Tensor:
    """
    baseline_full:
        MSE over the whole output.

    physics_limited:
        MSE only over mask points.
    """
    label = align_channels(label, out)

    if training_mode == "baseline_full":
        return crit(out, label)

    if training_mode == "physics_limited":
        if mask is None:
            raise ValueError("physics_limited mode requires a mask.")
        mask_c = expand_mask_for_channels(mask.bool(), out).float()
        return mse_on_region(out, label, mask_c)

    raise ValueError(f"Unknown training_mode: {training_mode}")


def unpack_batch(batch, training_mode: TrainingMode):
    if training_mode == "physics_limited":
        feat, label, mask = batch
        return feat, label, mask
    if training_mode == "baseline_full":
        feat, label = batch
        return feat, label, None
    raise ValueError(f"Unknown training_mode: {training_mode}")


# Evaluation
# ----------------------------------


def evaluate_loader(model: nn.Module, loader: DataLoader, config: ExperimentConfig, crit: nn.Module) -> Dict[str, float]:
    model.eval()

    total_mse = 0.0
    supervised_mse = 0.0
    mask_mse = 0.0
    nonmask_mse = 0.0
    darcy = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in loader:
            feat, label, mask = unpack_batch(batch, config.training_mode)
            feat = feat.to(DEVICE)
            label = label.to(DEVICE)
            mask = mask.to(DEVICE).bool() if mask is not None else None

            out = model(feat)
            label = align_channels(label, out)

            total_mse += crit(out, label).item()
            supervised_mse += supervised_loss(
                out, label, mask, config.training_mode, crit
            ).item()
            darcy += darcy_loss_from_output(out).item()

            if mask is not None:
                mask_c = expand_mask_for_channels(mask, out)
                nonmask_c = ~mask_c
                mask_mse += mse_on_region(out, label, mask_c).item()
                nonmask_mse += mse_on_region(out, label, nonmask_c).item()
            else:
                mask_mse += float("nan")
                nonmask_mse += float("nan")

            n_batches += 1

    return {
        "total_mse": total_mse / n_batches,
        "supervised_mse": supervised_mse / n_batches,
        "mask_mse": mask_mse / n_batches,
        "nonmask_mse": nonmask_mse / n_batches,
        "darcy": darcy / n_batches,
    }


# Training
# ------------------------------------


def run_experiment(**kwargs):
    """
    Physics run:
        model, history = run_experiment(
            model_type="splitnet_attn",
            dataset_mode="fixed",
            training_mode="physics_limited",
            mse_weight=1.0,
            darcy_weight=0.1,
            epochs=250,
            save_prefix="minimum_info"
        )

    No physics baseline:
        model, history = run_experiment(
            model_type="splitnet_attn",
            dataset_mode="fixed",
            training_mode="baseline_full",
            mse_weight=1.0,
            darcy_weight=0.0,
            epochs=250,
            save_prefix="minimum_info"
        )
    """
    config = ExperimentConfig(**kwargs)

    if config.training_mode == "baseline_full" and config.darcy_weight != 0:
        raise ValueError("baseline_full is the no physics baseline. Set darcy_weight=0.0, ")

    path_prefix = make_run_prefix(config)
    ensure_parent_dir(path_prefix)

    model = make_model(config.model_type, config.channels)
    optimizer = Adam(model.parameters(), lr=config.lr)
    crit = nn.MSELoss()

    train_loader, val_loader = make_loaders(config)

    history = {
        "train_loss_used": [],
        "train_total_mse": [],
        "train_supervised_mse": [],
        "train_mask_mse": [],
        "train_nonmask_mse": [],
        "train_darcy": [],
        "val_total_mse": [],
        "val_supervised_mse": [],
        "val_mask_mse": [],
        "val_nonmask_mse": [],
        "val_darcy": [],
        "config": asdict(config),
    }

    best_val_score = float("inf")
    best_epoch = 0

    for epoch in tqdm(range(1, config.epochs + 1)):
        model.train()
        epoch_loss = 0.0
        n_batches = 0

        for batch in train_loader:
            feat, label, mask = unpack_batch(batch, config.training_mode)
            feat = feat.to(DEVICE)
            label = label.to(DEVICE)
            mask = mask.to(DEVICE).bool() if mask is not None else None

            optimizer.zero_grad()

            out = model(feat)
            label = align_channels(label, out)

            mse = supervised_loss(out, label, mask, config.training_mode, crit)
            physics = darcy_loss_from_output(out)

            loss = config.mse_weight * mse + config.darcy_weight * physics
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        history["train_loss_used"].append(epoch_loss / n_batches)

        train_metrics = evaluate_loader(model, train_loader, config, crit)
        val_metrics = evaluate_loader(model, val_loader, config, crit)

        history["train_total_mse"].append(train_metrics["total_mse"])
        history["train_supervised_mse"].append(train_metrics["supervised_mse"])
        history["train_mask_mse"].append(train_metrics["mask_mse"])
        history["train_nonmask_mse"].append(train_metrics["nonmask_mse"])
        history["train_darcy"].append(train_metrics["darcy"])

        history["val_total_mse"].append(val_metrics["total_mse"])
        history["val_supervised_mse"].append(val_metrics["supervised_mse"])
        history["val_mask_mse"].append(val_metrics["mask_mse"])
        history["val_nonmask_mse"].append(val_metrics["nonmask_mse"])
        history["val_darcy"].append(val_metrics["darcy"])

        # Model selection metric:
        # supervised_mse is the validation version of the actual training MSE.
        if config.best_metric is not None:
            val_score = val_metrics[config.best_metric]
        elif config.training_mode == "physics_limited":
            val_score = val_metrics["supervised_mse"]
        else:
            val_score = val_metrics["total_mse"]

        if val_score < best_val_score:
            best_val_score = val_score
            best_epoch = epoch

            if config.save_best:
                ensure_parent_dir(path_prefix)
                torch.save(model.state_dict(), f"{path_prefix}_best_state.pt")

    metric_name = config.best_metric or ("val_supervised_mse" if config.training_mode == "physics_limited" else "val_total_mse")
    print(f"Best epoch: {best_epoch}, best {metric_name}: {best_val_score:.6f}")

    save_run_outputs(path_prefix, model, history, config, best_epoch, best_val_score)

    return model, history


# Convenience helper for physics weight sweeps
# -----------------------------------------------------------------------------



def run_experiment_grid(
    model_types=("splitnet_attn",),
    dataset_modes=("fixed",),
    training_modes=("physics_limited",),
    darcy_weights=(1.0,),
    epochs=250,
    base_save_dir="results",
    **kwargs,
):
    """
    General experiment grid runner.

    Runs every combination of:
        model_type × dataset_mode × training_mode × darcy_weight

    For baseline_full, darcy_weight is forced to 0.0.
    """

    results = {}

    for dataset_mode in dataset_modes:
        for model_type in model_types:
            for training_mode in training_modes:

                # baseline_full should always use no Darcy.
                weights_to_run = [0.0] if training_mode == "baseline_full" else darcy_weights

                for darcy_weight in weights_to_run:
                    safe_w = str(darcy_weight).replace(".", "p")

                    if training_mode == "baseline_full":
                        run_name = f"{dataset_mode}_{model_type}_baseline_full_nodarcy"
                    else:
                        run_name = (
                            f"{dataset_mode}_{model_type}_"
                            f"physics_limited_darcy_{safe_w}"
                        )

                    print(f"\n===== Running {run_name} =====")

                    model, history = run_experiment(
                        model_type=model_type,
                        dataset_mode=dataset_mode,
                        training_mode=training_mode,
                        mse_weight=1.0,
                        darcy_weight=darcy_weight,
                        epochs=epochs,
                        save_dir=base_save_dir,
                        run_name=run_name,
                        **kwargs,
                    )

                    results[run_name] = {
                        "model": model,
                        "history": history,
                        "best_val_total_mse": min(history["val_total_mse"]),
                        "best_val_supervised_mse": min(history["val_supervised_mse"]),
                        "final_val_total_mse": history["val_total_mse"][-1],
                        "final_val_supervised_mse": history["val_supervised_mse"][-1],
                        "final_val_darcy": history["val_darcy"][-1],
                    }

    return results